In [ ]:
# TITULO: Entrenamiento Comparativo - Detector de Placas
import os
from ultralytics import YOLO
from datetime import datetime
import pandas as pd
import matplotlib.pyplot as plt

# Rutas de los datasets generados
YAML_CLEAN = '../../datasets/02_placas/data.yaml'
# YAML_BASELINE = '../../datasets/02_placas_baseline/data.yaml'

# Ruta de salida de modelos
MODELS_DIR = '../../../../models/02_placas'

# Fecha para versionado
DATE_STR = datetime.now().strftime('%Y%m%d')

print("Configuracion lista.")

Configuracion lista.


In [2]:
# Función para obtener el optimizador real cuando se usa 'auto'
def obtener_optimizador_real(modelo):
    """
    Recupera el nombre real del optimizador cuando la configuración es 'auto'.
    """
    try:
        # 1. Si el entrenamiento acaba de terminar y el objeto sigue en memoria RAM
        if hasattr(modelo, 'trainer') and modelo.trainer and hasattr(modelo.trainer, 'optimizer'):
            # El optimizador es un objeto (ej. <torch.optim.sgd.SGD object at 0x...>)
            # Obtenemos su nombre de clase real
            opt_obj = modelo.trainer.optimizer
            nombre_real = type(opt_obj).__name__
            
            # También podemos sacar el Learning Rate real final
            lr_final = opt_obj.param_groups[0]['lr']
            
            print(f"Decisión de 'Auto':")
            print(f"   • Optimizador:   {nombre_real}") # Dirá 'SGD' o 'AdamW'
            print(f"   • Learning Rate: {lr_final:.6f}")
            return

        # 2. Si el modelo fue cargado desde disco (.pt) y no hay trainer en memoria
        # Buscamos en los metadatos internos del archivo
        if hasattr(modelo, 'ckpt') and modelo.ckpt:
            train_args = modelo.ckpt.get('train_args', {})
            # A veces aquí también dice 'auto', en cuyo caso la única verdad está en los logs de texto
            print(f"Configuración guardada: {train_args.get('optimizer', 'Desconocido')}")
            print("Si aquí dice 'auto', por favor revisa el archivo '/runs/.../train/main.log'")

    except Exception as e:
        print(f"No se pudo recuperar automáticamente: {e}")

In [3]:
# Entrenamiento YOLOv11n con dataset Resplit
run_name_v11n_resplit_tl = f"101_v11n_resplit_tl"

print(f"Iniciando entrenamiento: {run_name_v11n_resplit_tl}")

# Cargar modelo Nano pre-entrenado
model_v11n_resplit_tl = YOLO('yolo11n.pt')

results_v11n_resplit_tl = model_v11n_resplit_tl.train(
    data=YAML_CLEAN,  # Dataset resplit
    model='yolo11n.pt',  # Modelo pre-entrenado
    project=MODELS_DIR,
    name=run_name_v11n_resplit_tl,

    epochs=300,            # Ajustable
    patience=50,          # Early stopping    
    batch=32,          # Ajustable
    imgsz=640,            # Tamaño de imagen    
    
    exist_ok=False,         # Sobrescribir si existe
    pretrained=True,
    optimizer='AdamW',
    verbose=True,
    workers=os.cpu_count(),
    close_mosaic=10 # Apaga aumentación durante las últimas 10 épocas.
)

Iniciando entrenamiento: 101_v11n_resplit_tl
New https://pypi.org/project/ultralytics/8.4.125 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.116 🚀 Python-3.10.20 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11876MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../datasets/02_placas/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, 

/home/robertoplr/miniconda3/envs/proyecto/lib/python3.10/site-packages/ultralytics/utils/tal.py:401: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:91.)
  sx = feats[0].new_full((w,), 1, dtype=dtype).cumsum(0) - (1 - grid_cell_offset)  # shift x
/home/robertoplr/miniconda3/envs/proyecto/lib/python3.10/site-packages/ultralytics/utils/tal.py:402: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:91.)
  sy = feat

AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 673.0±108.8 MB/s, size: 2707.2 KB)
train: Scanning /home/robertoplr/Documentos/moca_proyecto/datasets/02_placas/train/labels... 794 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 794/794 3.3Kit/s 0.2s0.1ss
train: New cache created: /home/robertoplr/Documentos/moca_proyecto/datasets/02_placas/train/labels.cache
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 473.7±195.5 MB/s, size: 2155.4 KB)
val: Scanning /home/robertoplr/Documentos/moca_proyecto/datasets/02_placas/val/labels.cache... 99 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 99/99 3.1Mit/s 0.0s
optimizer: AdamW(lr=0.01, momentum=0.937) with parameter groups 81 weight(decay=0.0), 88 weight(decay=0.0005), 87 bias(decay=0.0)
Plotting labels to /home/robertoplr/Documentos/moca_proyecto/notebooks/02_placas/models/02_placas/101_v11n_resplit_tl/labels.jpg... 
WARNING ⚠️ 
Image sizes 640 train, 640 val
Using 12 dataloader workers
Logging results to /h

RuntimeError: DataLoader worker (pid(s) 4924, 4936, 4948, 4960, 4972, 4984, 4996, 5008, 5020, 5032, 5044, 5056) exited unexpectedly

In [9]:
# --- RECUPERAR HIPERPARÁMETROS REALES ---
# Ejecutar al finalizar el entrenamiento
print(f"Batch Size: {model_v11n_resplit_tl.trainer.args.batch}") 
print(f"Optimizer: {obtener_optimizador_real(model_v11n_resplit_tl)}")
print(f"Learning Rate inicial: {model_v11n_resplit_tl.trainer.args.lr0}")

Batch Size: 3
Decisión de 'Auto':
   • Optimizador:   AdamW
   • Learning Rate: 0.000027
Optimizer: None
Learning Rate inicial: 0.01


In [ ]:
print("Validando modelo 100_v11n_resplit_tl en split='test'...")

PROJECT_DIR = '../../../../models/02_placas'
run_name = f"100_v11n_resplit_tl"

# Cargar el MEJOR modelo resultante del entrenamiento anterior
model_yolov8n_baseline_tl = os.path.join(PROJECT_DIR, run_name, 'weights', 'best.pt')
best_model = YOLO(model_yolov8n_baseline_tl)

# Ejecutar validación en split='test'
metrics = best_model.val(
    split='test', 
    project=PROJECT_DIR, 
    name=f"{run_name}_eval", 
    imgsz=640,      # Tamaño de imagen
    batch=49,        # Mismo batch que entrenamiento
    plots=True       # Generar gráficos de métricas
)

print(f"\nResultados Finales en Test del dataset Baseline:")
print(f"   mAP@50:    {metrics.box.map50:.4f} (Precisión holgada)")
print(f"   mAP@50-95: {metrics.box.map:.4f}  (Precisión estricta )")

Validando modelo 100_v11n_resplit_tl en split='test'...
Ultralytics 8.4.116 🚀 Python-3.10.20 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11873MiB)
YOLO11n summary (fused): 101 layers, 2,582,347 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 8227.3±1335.2 MB/s, size: 1821.6 KB)
val: Scanning /home/robertoplr/Documentos/moca_proyecto/datasets/02_placas/test/labels... 100 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 100/100 3.2Kit/s 0.0s
val: New cache created: /home/robertoplr/Documentos/moca_proyecto/datasets/02_placas/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 2.0s/it 5.9s1.3s3s
                   all        100        123      0.983      0.967      0.974       0.85
Speed: 2.9ms preprocess, 1.7ms inference, 0.0ms loss, 0.3ms postprocess per image
Results saved to /home/robertoplr/Documentos/moca_proyecto/notebooks/02_placas/models/02_placas/100_

In [ ]:
import glob
import random

run_name = f"100_v11n_resplit_tl_inference"

# Tomar una imagen de prueba aleatoria
test_images = glob.glob('../../datasets/02_placas/test/images/*.jpg')
if test_images:
    sample_img = random.choice(test_images)
    
    # Prediccion
    res = model_v11n_resplit_tl.predict(sample_img, save=True, project=MODELS_DIR, name=run_name)
     
    print(f"Inferencia guardada en {res[0].save_dir}")
else:
    print("No se encontraron imagenes de prueba.")


image 1/1 /home/robertoplr/Documentos/moca_proyecto/notebooks/02_placas/../../datasets/02_placas/test/images/01768.jpg: 864x1280 1 license_plate, 4.9ms
Speed: 6.1ms preprocess, 4.9ms inference, 0.7ms postprocess per image at shape (1, 3, 864, 1280)
Results saved to /home/robertoplr/Documentos/moca_proyecto/models/02_placas/20260808_v11n_resplit_tl_inference
Inferencia guardada en /home/robertoplr/Documentos/moca_proyecto/models/02_placas/20260808_v11n_resplit_tl_inference


In [ ]:
# MODELS_DIR = '../../models/02_placas'
# Cargar mejores pesos
# Asegúrate que estos nombres coincidan exactamente con los definidos en las celdas de entrenamiento
run_name_v11 = f"100_v11n_resplit_tl"

path_v11_weights = os.path.join(MODELS_DIR, run_name_v11, 'weights', 'best.pt')

model_final_v11 = YOLO(path_v11_weights)

print("--- VALIDACION CRUZADA ---")

# 1. Validar Modelo YOLOv11n
metrics_v11 = model_final_v11.val(
    data=YAML_CLEAN, 
    split='test', 
    project=MODELS_DIR,
    name=f"{run_name_v11}_val"
)

print("\nRESULTADOS COMPARATIVOS (mAP50-95):")
print(f"Modelo YOLOv11n: {metrics_v11.box.map:.4f}")

--- VALIDACION CRUZADA ---
Ultralytics 8.4.116 🚀 Python-3.10.20 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11873MiB)
YOLO11n summary (fused): 101 layers, 2,582,347 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 10365.6±1889.9 MB/s, size: 2368.4 KB)
val: Scanning /home/robertoplr/Documentos/moca_proyecto/datasets/02_placas/test/labels.cache... 100 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 100/100 20.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 1.8it/s 4.0s0.2ss
                   all        100        123          1      0.983      0.995       0.92
Speed: 7.9ms preprocess, 4.8ms inference, 0.0ms loss, 0.6ms postprocess per image
Results saved to /home/robertoplr/Documentos/moca_proyecto/notebooks/02_placas/models/02_placas/100_v11n_resplit_tl_val

RESULTADOS COMPARATIVOS (mAP50-95):
Modelo YOLOv11n: 0.9204


In [14]:
print("\nINFORMACION DEL MODELO YOLOv11n:")
# print(model_final_v11.info)
model_final_v11.info()


INFORMACION DEL MODELO YOLOv11n:
YOLO11n summary (fused): 101 layers, 2,582,347 parameters, 0 gradients, 6.4 GFLOPs


(101, 2582347, 0, 6.3719936)

In [16]:
from torchinfo import summary

def desplegar_arquitectura_completa(modelo_yolo, input_size=(1, 3, 640, 640)):
    """
    Muestra el resumen completo de capas, params y tamaños de memoria.
    input_size: (Batch, Canales, Alto, Ancho)
    """
    print(f"\n🔍 ARQUITECTURA DETALLADA: {modelo_yolo.task_map}")
    # Accedemos al modelo interno de PyTorch (modelo.model)
    summary(modelo_yolo.model, 
            input_size=input_size, 
            col_names=["input_size", "output_size", "num_params", "kernel_size", "mult_adds"],
            verbose=1)

In [1]:
desplegar_arquitectura_completa(model_final_v11, input_size=(1, 3, 640, 640))

NameError: name 'desplegar_arquitectura_completa' is not defined

In [8]:
from ultralytics import YOLO

# Cargar TU modelo previamente entrenado en lugar del modelo base de Ultralytics
modelo = YOLO("../../models/02_placas/100_v11n_resplit_tl/weights/best.pt")

# Iniciar el ajuste fino con hiperparámetros restrictivos
resultados = modelo.train(
    data = '../../datasets/02_placas/data.yaml',
    epochs=150,
    patience=25,
    batch=16,
    imgsz=640,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    warmup_epochs=5,
    cos_lr=True
)

New https://pypi.org/project/ultralytics/8.4.120 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.116 🚀 Python-3.10.20 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11876MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=../../datasets/02_placas/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300

/home/robertoplr/miniconda3/envs/proyecto/lib/python3.10/site-packages/ultralytics/utils/tal.py:401: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:91.)
  sx = feats[0].new_full((w,), 1, dtype=dtype).cumsum(0) - (1 - grid_cell_offset)  # shift x
/home/robertoplr/miniconda3/envs/proyecto/lib/python3.10/site-packages/ultralytics/utils/tal.py:402: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:91.)
  sy = feat

AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 10348.7±2254.5 MB/s, size: 2898.0 KB)
train: Scanning /home/robertoplr/Documentos/moca_proyecto/datasets/02_placas/train/labels... 794 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 794/794 3.3Kit/s 0.2s0.1ss
train: New cache created: /home/robertoplr/Documentos/moca_proyecto/datasets/02_placas/train/labels.cache
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 586.7±183.7 MB/s, size: 2444.0 KB)
val: Scanning /home/robertoplr/Documentos/moca_proyecto/datasets/02_placas/val/labels.cache... 99 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 99/99 3.7Mit/s 0.0s
optimizer: AdamW(lr=0.001, momentum=0.937) with parameter groups 81 weight(decay=0.0), 88 weight(decay=0.0005), 87 bias(decay=0.0)
Plotting labels to /home/robertoplr/Documentos/moca_proyecto/notebooks/02_placas/runs/detect/train-2/labels.jpg... 
Image sizes 640 train, 640 val
Using 8 dataloader workers
Logging results to /home/robertoplr/Documentos/

/home/robertoplr/miniconda3/envs/proyecto/lib/python3.10/site-packages/ultralytics/utils/tal.py:401: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:91.)
  sx = feats[0].new_full((w,), 1, dtype=dtype).cumsum(0) - (1 - grid_cell_offset)  # shift x
/home/robertoplr/miniconda3/envs/proyecto/lib/python3.10/site-packages/ultralytics/utils/tal.py:402: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:91.)
  sy = feat


      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      1/150      2.28G      1.043     0.7135     0.8844         24        640: 100% ━━━━━━━━━━━━ 50/50 2.9it/s 17.3s0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 4.6it/s 0.9s0.3s
                   all         99        115      0.932       0.87      0.907      0.739

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      2/150      2.59G      1.011     0.7014     0.8652         28        640: 0% ──────────── 0/50  0.1s

...

     68/150      2.59G       0.51     0.3126     0.8022         12        640: 100% ━━━━━━━━━━━━ 50/50 3.6it/s 14.0s0.3ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 12.9it/s 0.3s.6s
                   all         99        115          1       0.98      0.994      0.854

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     69/150      2.59G     0.5248     0.3308     0.7912         40        640: 2% ──────────── 1/50 3.0it/s 0.2s<16.5s

/home/robertoplr/miniconda3/envs/proyecto/lib/python3.10/site-packages/ultralytics/utils/tal.py:401: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:91.)
  sx = feats[0].new_full((w,), 1, dtype=dtype).cumsum(0) - (1 - grid_cell_offset)  # shift x
/home/robertoplr/miniconda3/envs/proyecto/lib/python3.10/site-packages/ultralytics/utils/tal.py:402: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:91.)
  sy = feat

     69/150      2.59G     0.5046     0.3169     0.7985         20        640: 100% ━━━━━━━━━━━━ 50/50 3.5it/s 14.5s0.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 12.4it/s 0.3s.5s
                   all         99        115      0.991      0.991      0.995      0.864

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     70/150      2.59G     0.5213     0.2858      0.782         32        640: 0% ──────────── 0/50  0.1s

/home/robertoplr/miniconda3/envs/proyecto/lib/python3.10/site-packages/ultralytics/utils/tal.py:401: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:91.)
  sx = feats[0].new_full((w,), 1, dtype=dtype).cumsum(0) - (1 - grid_cell_offset)  # shift x
/home/robertoplr/miniconda3/envs/proyecto/lib/python3.10/site-packages/ultralytics/utils/tal.py:402: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:91.)
  sy = feat

     70/150      2.59G     0.5268     0.3181     0.7924         26        640: 100% ━━━━━━━━━━━━ 50/50 3.6it/s 13.7s<0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 12.2it/s 0.3s.6s
                   all         99        115      0.998      0.991      0.995      0.865
EarlyStopping: Training stopped early as no improvement observed in last 25 epochs. Best results observed at epoch 45, best model saved as best.pt.
To update EarlyStopping(patience=25) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

70 epochs completed in 0.293 hours.
Optimizer stripped from /home/robertoplr/Documentos/moca_proyecto/notebooks/02_placas/runs/detect/train-2/weights/last.pt, 5.5MB
Optimizer stripped from /home/robertoplr/Documentos/moca_proyecto/notebooks/02_placas/runs/detect/train-2/weights/best.pt, 5.5MB

Validating /home/robertoplr/Documentos/moca_proyecto/notebooks/02_placas/runs/detect/t

In [1]:
from ultralytics import YOLO
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns



def evaluar_modelos(ruta_yaml, ruta_modelo_original, ruta_modelo_nuevo):
    print("Iniciando evaluación del Modelo Original...")
    modelo_orig = YOLO(ruta_modelo_original)
    # Ejecutamos validación silenciando la salida excesiva
    val_orig = modelo_orig.val(data=ruta_yaml, verbose=False)
    
    print("Iniciando evaluación del Modelo Fine-Tuned (Sintético)...")
    modelo_nuevo = YOLO(ruta_modelo_nuevo)
    val_nuevo = modelo_nuevo.val(data=ruta_yaml, verbose=False)
    
    # Extraer métricas (Clase 0 general)
    # Resultados de Ultralytics val: results.box.map50, results.box.map, results.box.p, results.box.r
    metricas = {
        "Métrica": ["Precisión", "Recall", "mAP@50", "mAP@50-95"],
        "Modelo Original": [
            val_orig.box.p[0], 
            val_orig.box.r[0], 
            val_orig.box.map50, 
            val_orig.box.map
        ],
        "Modelo Sintético": [
            val_nuevo.box.p[0], 
            val_nuevo.box.r[0], 
            val_nuevo.box.map50, 
            val_nuevo.box.map
        ]
    }
    
    # 1. Crear y mostrar tabla comparativa con Pandas
    df_metricas = pd.DataFrame(metricas)
    
    # Calcular la mejora porcentual
    df_metricas["Mejora Absoluta"] = df_metricas["Modelo Sintético"] - df_metricas["Modelo Original"]
    
    print("\n" + "="*50)
    print("TABLA COMPARATIVA DE RENDIMIENTO")
    print("="*50)
    print(df_metricas.round(4).to_string(index=False))
    print("="*50 + "\n")
    
    # 2. Generar gráfico visual con Seaborn y Matplotlib
    # Reestructurar el DataFrame (Melt) para Seaborn
    df_melted = df_metricas.melt(id_vars=["Métrica"], 
                                 value_vars=["Modelo Original", "Modelo Sintético"], 
                                 var_name="Modelo", 
                                 value_name="Puntuación")
    
    plt.figure(figsize=(10, 6))
    sns.set_theme(style="whitegrid")
    
    grafico = sns.barplot(
        data=df_melted, 
        x="Métrica", 
        y="Puntuación", 
        hue="Modelo", 
        palette=["#E74C3C", "#2ECC71"] # Rojo para original, Verde para nuevo
    )
    
    plt.title('Comparativa de Rendimiento YOLOv11: Original vs Sintético', fontsize=14, fontweight='bold')
    plt.ylim(0, 1.1) # Rango de 0 a 1 para las métricas
    plt.ylabel('Puntuación (0.0 - 1.0)', fontsize=12)
    plt.xlabel('Métrica Evaluada', fontsize=12)
    
    # Añadir las etiquetas de datos sobre cada barra
    for p in grafico.patches:
        grafico.annotate(format(p.get_height(), '.3f'), 
                         (p.get_x() + p.get_width() / 2., p.get_height()), 
                         ha = 'center', va = 'center', 
                         xytext = (0, 9), 
                         textcoords = 'offset points',
                         fontsize=10)
                         
    plt.tight_layout()
    plt.savefig('comparativa_modelos_moca.png', dpi=300)
    print("Gráfico guardado exitosamente como 'comparativa_modelos_moca.png'.")
    plt.show()

# ==========================================
# EJECUCIÓN
# ==========================================

archivo_yaml = "../../datasets/02_placas/data.yaml"
pesos_originales = "../../production_weights/02_placas_best_anterior.pt"
pesos_nuevos = "../../production_weights/02_placas_best.pt"

evaluar_modelos(archivo_yaml, pesos_originales, pesos_nuevos)

Iniciando evaluación del Modelo Original...
Ultralytics 8.4.116 🚀 Python-3.10.20 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11876MiB)
YOLO11n summary (fused): 101 layers, 2,582,347 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 12788.1±4447.4 MB/s, size: 2037.8 KB)
val: Scanning /home/robertoplr/Documentos/moca_proyecto/datasets/02_placas/val/labels.cache... 99 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 99/99 27.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 1.8it/s 3.9s0.2ss
                   all         99        115      0.999      0.991      0.995      0.932
Speed: 5.7ms preprocess, 5.3ms inference, 0.0ms loss, 0.6ms postprocess per image
Results saved to /home/robertoplr/Documentos/moca_proyecto/notebooks/02_placas/runs/detect/val
Iniciando evaluación del Modelo Fine-Tuned (Sintético)...
Ultralytics 8.4.116 🚀 Python-3.10.20 torch-2.5.1+cu121 CUDA

<Figure size 1000x600 with 1 Axes>